# Task Complexity Measurement Pipeline
### Structural & Linguistic Feature Approach — Step by Step

---

**What this notebook does:**
1. Load a dataset (Parquet / CSV format)
2. Extract structural & linguistic complexity features for every task
3. Combine features into a single normalised complexity score `[0.0 → 1.0]`
4. Label each task: **Easy / Medium / Hard**
5. Visualise the results and export enriched Parquet

**Features measured:**
| Feature | What it captures |
|---|---|
| `token_count` | Raw length — longer = harder |
| `compression_ratio` | Kolmogorov proxy — less compressible = richer info |
| `type_token_ratio` | Lexical diversity — more unique words = harder |
| `sentence_count` | More sentences = more context / steps |
| `num_reasoning_steps` | Explicit step indicators (*first, then, finally…*) |
| `decision_branch_depth` | Conditional branches (*if, unless, given that…*) |
| `cross_domain_score` | Multi-domain vocabulary hit count |
| `ambiguity_score` | Underspecified / vague language |
| `question_density` | How many sub-questions are embedded |
| `hop_signal_count` | Multi-hop connectors (*according to, therefore…*) |

> **Run each cell from top to bottom. Every cell is independent and fully explained.**

---
## STEP 0 — Imports & Setup
Only standard library + pandas + numpy. No GPU, no API keys needed.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import re
import gzip
import math
import json
import statistics
import collections
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

# ── Third-party ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = Path('../datasets/processed/gaia.parquet')   # ← Path object, not pd.read_parquet()
OUTPUT_DIR = Path('data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Imports OK')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   Data    {DATA_PATH}')

✅ Imports OK
   pandas  3.0.1
   numpy   2.4.3
   Data                                           id  \
0    c61d22de-5f6c-4958-a7f6-5e9707bd3466   
1    17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc   
2    04a04a9b-226c-43fd-b319-d5e89743676f   
3    14569e28-c88c-43e4-8c32-097d35b9a67d   
4    e1fc63a2-da7a-432f-be78-7c4a95598703   
..                                    ...   
160  0bdb7c40-671d-4ad1-9ce3-986b159c0ddc   
161  08c0b6e9-1b43-4c2e-ae55-4e3fce2c2715   
162  db4fd70a-2d37-40ea-873f-9433dc5e301f   
163  853c8244-429e-46ca-89f2-addf40dfb2bd   
164  7a4a336d-dcfa-45a0-b014-824c7619e8de   

                                                 query         answer level  \
0    A paper about AI regulation that was originall...    egalitarian     2   
1    I’m researching species that became invasive a...          34689     2   
2    If we assume all articles published by Nature ...             41     2   
3    In Unlambda, what exact charcter or text needs...       backtick     2   
4    I

---
## STEP 1 — Load Dataset from Parquet / CSV

The loader automatically detects `.parquet` or `.csv` format.
It also validates the required columns so you get a clear error
if something is wrong — before doing any expensive computation.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Dataset Loader
# Handles: .parquet, .csv, .jsonl  |  column remapping  |  validates schema
# ─────────────────────────────────────────────────────────────────────────────

REQUIRED_COLUMNS = ['id', 'query', 'task_type']

# ── Column remapping table ────────────────────────────────────────────────────
# Maps each dataset's native column names → our standard schema.
# Add a new entry here whenever you onboard a new dataset.
COLUMN_REMAP: dict[str, dict[str, str]] = {
    'gaia': {
        'task_id':      'id',
        'Question':     'query',
        'Level':        'task_type',          # GAIA levels 1/2/3 act as task_type
        'Final answer': 'reference_answer',
        'file_name':    'attached_file',
    },
    'mmlu_pro': {
        'question_id':  'id',
        'question':     'query',
        'category':     'task_type',
        'answer':       'reference_answer',
    },
    'musique': {
        'id':           'id',
        'question':     'query',
        'answer':       'reference_answer',
    },
    'aime': {
        'ID':           'id',
        'Problem':      'query',
        'Answer':       'reference_answer',
    },
    'swe_bench': {
        'instance_id':        'id',
        'problem_statement':  'query',
        'repo':               'task_type',
        'patch':              'reference_answer',
    },
}


def _detect_dataset(path: Path) -> str:
    """
    Infer dataset name from the file path stem.
    e.g.  '../datasets/processed/gaia.parquet'  →  'gaia'
    """
    stem = path.stem.lower()
    for known in COLUMN_REMAP:
        if known in stem:
            return known
    return 'unknown'


def _apply_remap(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """
    Rename native columns to the standard schema using COLUMN_REMAP.
    Only renames columns that actually exist — ignores missing ones silently.
    """
    if dataset_name not in COLUMN_REMAP:
        return df                               # no mapping defined — pass through

    remap = {
        src: dst
        for src, dst in COLUMN_REMAP[dataset_name].items()
        if src in df.columns                    # only rename what exists
    }
    df = df.rename(columns=remap)

    # If task_type still missing after remap, fill with dataset name as fallback
    if 'task_type' not in df.columns:
        df['task_type'] = dataset_name

    # If id still missing, create one from row index
    if 'id' not in df.columns:
        df['id'] = [f'{dataset_name}_{i:05d}' for i in range(len(df))]

    return df


def load_dataset(path: Path) -> pd.DataFrame:
    """
    Load a benchmark dataset from Parquet, CSV, or JSONL.
    Automatically remaps column names to the standard schema.

    Parameters
    ----------
    path : Path to the file. Extension determines the loader used.

    Returns
    -------
    pd.DataFrame guaranteed to contain: id, query, task_type

    Raises
    ------
    FileNotFoundError  if the file does not exist.
    ValueError         if required columns are missing after remapping.
    ImportError        if pyarrow is not installed (Parquet only).
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    suffix       = path.suffix.lower()
    dataset_name = _detect_dataset(path)

    # ── 1. Read raw file ──────────────────────────────────────────────────────
    if suffix == '.parquet':
        try:
            df = pd.read_parquet(path)
            print(f'📦 Loaded Parquet : {path.name}')
        except ImportError:
            raise ImportError(
                'pyarrow is required to read Parquet files.\n'
                'Install it with:  pip install pyarrow'
            )

    elif suffix == '.csv':
        df = pd.read_csv(path, low_memory=False)
        print(f'📄 Loaded CSV     : {path.name}')

    elif suffix == '.jsonl':
        records = []
        with path.open('r', encoding='utf-8') as fh:
            for line in fh:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        df = pd.DataFrame(records)
        print(f'📝 Loaded JSONL   : {path.name}  ({len(records):,} lines)')

    else:
        raise ValueError(
            f'Unsupported format: "{suffix}". '
            'Supported formats are: .parquet  .csv  .jsonl'
        )

    print(f'   Detected dataset : {dataset_name}')
    print(f'   Raw columns      : {list(df.columns)}')

    # ── 2. Remap columns to standard schema ───────────────────────────────────
    df = _apply_remap(df, dataset_name)

    # ── 3. Validate required columns ──────────────────────────────────────────
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f'Missing required columns after remapping: {missing}\n'
            f'Columns present: {list(df.columns)}\n'
            f'Tip: add a mapping for "{dataset_name}" to COLUMN_REMAP above.'
        )

    # ── 4. Basic cleanup ──────────────────────────────────────────────────────
    df['query']     = df['query'].fillna('').astype(str)
    df['id']        = df['id'].astype(str)
    df['task_type'] = df['task_type'].astype(str)
    df = df[df['query'].str.strip() != '']      # drop empty queries
    df = df.reset_index(drop=True)

    return df


# ── Run ───────────────────────────────────────────────────────────────────────
df = load_dataset(DATA_PATH)

print(f'\n📊 Dataset shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Final columns   : {list(df.columns)}')
print(f'\n   task_type breakdown:')
print(df['task_type'].value_counts().to_string())
print(f'\n   First 3 queries (truncated to 80 chars):')
for _, row in df.head(3).iterrows():
    print(f'   [{row["task_type"]:12}]  {row["query"][:80]}…')

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'DataFrame'

---
## STEP 2 — Feature: Number of Reasoning Steps

**What it measures:** How many explicit sequential steps the query implies.

**How it works:** We count *step-signalling words* — words that indicate
sequential reasoning (`first`, `then`, `next`, `finally`, `step 1`, etc.).
A query with 5 step signals almost certainly requires a 5-step solution.

**Intuition:**
- `"What is 2+2?"` → 0 step signals → step_score ≈ 0.0
- `"First find x, then substitute into the equation, finally solve for y"` → 3 → step_score ≈ 0.45

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Feature: Reasoning Step Count
# ─────────────────────────────────────────────────────────────────────────────

# Patterns that signal an explicit sequential step in the query.
# Each pattern gets a weight: strong indicators ("step 1") > weak ("first").
STEP_PATTERNS = [
    # Strong signals — numbered steps or explicit 'step N'
    (r'\bstep\s+\d+\b',                        2.0),
    (r'\b(first|second|third|fourth|fifth)\b',  1.5),
    (r'\b(1st|2nd|3rd|4th|5th)\b',             1.5),
    (r'\bpart\s+[a-e\d]\b',                     1.5),
    # Medium signals — sequential connectors
    (r'\b(then|next|after that|subsequently)\b', 1.0),
    (r'\b(finally|lastly|in the end)\b',         1.0),
    (r'\b(additionally|furthermore|moreover)\b', 0.8),
    # Weak signals — parallel listing
    (r'\b(and also|as well as)\b',              0.5),
]

# Pre-compile for speed (important at scale)
_STEP_COMPILED = [(re.compile(p, re.IGNORECASE), w) for p, w in STEP_PATTERNS]


def count_reasoning_steps(text: str) -> Tuple[float, int]:
    """
    Count weighted step signals in the text.
    
    Returns
    -------
    raw_count  : Weighted sum of step signals found
    match_count: Number of distinct pattern matches (unweighted)
    """
    raw_count = 0.0
    match_count = 0
    
    for pattern, weight in _STEP_COMPILED:
        matches = pattern.findall(text)
        raw_count   += len(matches) * weight
        match_count += len(matches)
    
    return raw_count, match_count


def normalise_step_score(raw_count: float, saturation: float = 8.0) -> float:
    """
    Map raw weighted count → [0.0, 1.0] using a sigmoid-like formula.
    
    At raw_count = saturation, score ≈ 0.63  (1 - 1/e)
    At raw_count = 2*saturation, score ≈ 0.86
    
    Why sigmoid and not linear?
        A query with 20 step words is not 2× harder than one with 10.
        Complexity has diminishing returns — the sigmoid captures this.
    """
    return 1.0 - math.exp(-raw_count / saturation)


# ── Apply to every row ────────────────────────────────────────────────────────
step_results = df['query'].apply(count_reasoning_steps)
df['step_raw']   = step_results.apply(lambda x: x[0])   # weighted count
df['step_hits']  = step_results.apply(lambda x: x[1])   # match count
df['feat_step_score'] = df['step_raw'].apply(normalise_step_score)

# ── Show results ──────────────────────────────────────────────────────────────
print('=== Feature: Reasoning Step Count ===')
print(f'\nScale: 0.0 = no steps | 1.0 = many explicit steps')
print()
display_df = df[['id', 'task_type', 'step_hits', 'feat_step_score']].copy()
display_df['feat_step_score'] = display_df['feat_step_score'].round(4)
display_df.columns = ['id', 'task_type', 'step_matches', 'step_score']
print(display_df.to_string(index=False))

print('\nMean step score by task type:')
print(df.groupby('task_type')['feat_step_score'].mean().round(3).sort_values(ascending=False).to_string())

NameError: name 'df' is not defined

---
## STEP 3 — Feature: Decision Tree Depth

**What it measures:** How many conditional branches are in the problem.

**Intuition:** A problem with `if`, `unless`, `given that`, `assuming`, `suppose`
means the solver must handle multiple cases — deeper decision tree = harder.

| Example | Branches | Score |
|---|---|---|
| `"What is 5 × 4?"` | 0 | 0.00 |
| `"If x > 0, find y"` | 1 | 0.18 |
| `"If x > 0 and y < 0, unless z = 0, assuming a ≠ b, find w"` | 3 | 0.45 |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Feature: Decision Branch Depth
# ─────────────────────────────────────────────────────────────────────────────

# Each pattern signals a conditional branch in the problem.
# Weights: syntactic 'if' is stronger than soft 'assuming'.
BRANCH_PATTERNS = [
    # Strong conditional operators
    (r'\bif\b',                                2.0),
    (r'\bunless\b',                            1.8),
    (r'\botherwise\b',                         1.5),
    (r'\belseif\b|\belse if\b',               1.8),
    # Assumption-setting (defines a case)
    (r'\bassuming\b',                          1.2),
    (r'\bgiven that\b',                        1.3),
    (r'\bsuppose\b|\bsupposing\b',            1.2),
    (r'\blet\s+\w+\s*(=|be)\b',              1.0),  # "let x = 3"
    # Case enumeration
    (r'\bcase\s+\d+\b|\bin\s+case\b',         1.5),
    (r'\bwhen\b',                              0.8),
    (r'\bfor\s+(all|each|every|any)\b',        0.9),
    # Contrast operators (implicit else-branches)
    (r'\bhowever\b|\byet\b|\bbut\b',          0.5),
]

_BRANCH_COMPILED = [(re.compile(p, re.IGNORECASE), w) for p, w in BRANCH_PATTERNS]


def measure_decision_depth(text: str) -> Tuple[float, int]:
    """
    Measure the total 'branch weight' and the number of branch-signalling matches.
    
    Returns
    -------
    branch_weight : Weighted sum of branch signals
    branch_count  : Number of matched patterns (unweighted)
    """
    branch_weight = 0.0
    branch_count  = 0
    
    for pattern, weight in _BRANCH_COMPILED:
        matches = pattern.findall(text)
        branch_weight += len(matches) * weight
        branch_count  += len(matches)
    
    return branch_weight, branch_count


def normalise_branch_score(weight: float, saturation: float = 6.0) -> float:
    """
    Map weighted branch count → [0.0, 1.0].
    Saturation at 6 weighted units (≈ 3 strong 'if' conditions).
    """
    return 1.0 - math.exp(-weight / saturation)


# ── Apply ─────────────────────────────────────────────────────────────────────
branch_results = df['query'].apply(measure_decision_depth)
df['branch_weight'] = branch_results.apply(lambda x: x[0])
df['branch_count']  = branch_results.apply(lambda x: x[1])
df['feat_branch_score'] = df['branch_weight'].apply(normalise_branch_score)

# ── Show ──────────────────────────────────────────────────────────────────────
print('=== Feature: Decision Tree Depth ===')
print(f'\nScale: 0.0 = no branches | 1.0 = many conditional cases')
print()

display_df2 = df[['id', 'task_type', 'branch_count', 'feat_branch_score']].copy()
display_df2['feat_branch_score'] = display_df2['feat_branch_score'].round(4)
display_df2.columns = ['id', 'task_type', 'branch_matches', 'branch_score']
print(display_df2.to_string(index=False))

print('\nMean branch score by task type:')
print(df.groupby('task_type')['feat_branch_score'].mean().round(3).sort_values(ascending=False).to_string())

---
## STEP 4 — Feature: Cross-Domain Dependencies

**What it measures:** How many different knowledge domains the query spans.

**Why it matters:** A question about *chemistry + physics + mathematics together*
is harder than one that only needs chemistry, because the solver must
integrate knowledge from multiple fields.

**Method:** We build a vocabulary of domain-specific keywords per domain.
We count how many *distinct domains* are mentioned, then how many
total domain-keyword hits occur.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Feature: Cross-Domain Dependencies
# ─────────────────────────────────────────────────────────────────────────────

# Domain keyword vocabulary.
# These are intentionally non-overlapping — each word belongs to one domain.
# Add more domains / keywords to improve coverage.
DOMAIN_VOCAB: Dict[str, List[str]] = {
    'mathematics': [
        'integral', 'derivative', 'theorem', 'proof', 'matrix', 'vector',
        'eigenvalue', 'polynomial', 'congruence', 'modular', 'combinatorics',
        'permutation', 'probability', 'distribution', 'calculus', 'algebra',
        'geometry', 'topology', 'fourier', 'laplace', 'differential equation',
        'prime', 'fibonacci', 'sequence', 'series', 'convergence', 'limit',
    ],
    'physics': [
        'force', 'energy', 'momentum', 'velocity', 'acceleration', 'mass',
        'particle', 'quantum', 'wave', 'photon', 'electron', 'neutron',
        'entropy', 'thermodynamics', 'relativity', 'electromagnetic',
        'gravitational', 'friction', 'torque', 'angular', 'inertia',
        'pressure', 'temperature', 'density', 'magnetic field', 'electric field',
    ],
    'chemistry': [
        'molecule', 'atom', 'element', 'compound', 'reaction', 'oxidation',
        'reduction', 'acid', 'base', 'pH', 'molar', 'stoichiometry',
        'equilibrium', 'catalyst', 'polymer', 'bond', 'valence', 'isotope',
        'titration', 'spectroscopy', 'chromatography', 'enzyme', 'protein',
    ],
    'computer_science': [
        'algorithm', 'complexity', 'runtime', 'recursion', 'graph', 'tree',
        'binary', 'sorting', 'hashing', 'pointer', 'memory', 'stack',
        'queue', 'dynamic programming', 'neural network', 'gradient',
        'backpropagation', 'transformer', 'attention', 'embedding', 'token',
        'compiler', 'operating system', 'concurrency', 'thread', 'database',
    ],
    'biology': [
        'cell', 'dna', 'rna', 'protein', 'gene', 'chromosome', 'mutation',
        'evolution', 'species', 'organism', 'metabolism', 'photosynthesis',
        'mitosis', 'meiosis', 'neuron', 'synapse', 'immune', 'antibody',
        'ecosystem', 'population', 'taxonomy', 'phenotype', 'genotype',
    ],
    'history_politics': [
        'war', 'treaty', 'parliament', 'constitution', 'revolution',
        'dynasty', 'empire', 'democracy', 'legislature', 'president',
        'prime minister', 'colonialism', 'sovereignty', 'amendment',
        'election', 'referendum', 'senate', 'republic', 'monarchy',
    ],
    'economics': [
        'gdp', 'inflation', 'interest rate', 'supply', 'demand', 'market',
        'capital', 'investment', 'fiscal', 'monetary', 'trade', 'tariff',
        'currency', 'exchange rate', 'utility', 'elasticity', 'marginal',
        'equilibrium price', 'oligopoly', 'monopoly', 'recession',
    ],
    'law': [
        'statute', 'jurisdiction', 'plaintiff', 'defendant', 'tort',
        'contract', 'liability', 'precedent', 'appeal', 'verdict',
        'legislation', 'regulation', 'compliance', 'copyright', 'patent',
        'trademark', 'intellectual property', 'habeas corpus', 'due process',
    ],
}

# Pre-compile: {domain: [compiled_pattern, ...]}
_DOMAIN_COMPILED: Dict[str, List] = {
    domain: [re.compile(r'\b' + re.escape(kw) + r'\b', re.IGNORECASE) for kw in keywords]
    for domain, keywords in DOMAIN_VOCAB.items()
}


def measure_cross_domain(text: str) -> Tuple[int, int, List[str]]:
    """
    Count distinct domains touched and total keyword hits.
    
    Returns
    -------
    n_domains    : Number of distinct domains with at least 1 hit
    total_hits   : Total keyword matches across all domains
    domains_found: List of domain names that were hit
    """
    n_domains = 0
    total_hits = 0
    domains_found = []
    
    for domain, patterns in _DOMAIN_COMPILED.items():
        domain_hits = sum(len(p.findall(text)) for p in patterns)
        if domain_hits > 0:
            n_domains  += 1
            total_hits += domain_hits
            domains_found.append(domain)
    
    return n_domains, total_hits, domains_found


def normalise_cross_domain_score(n_domains: int, total_hits: int) -> float:
    """
    Combine distinct domains and hit density into [0, 1].
    
    Formula:
        domain_diversity  = n_domains / 4  (cap at 4 domains → 1.0)
        keyword_density   = 1 - exp(-total_hits / 6)
        final             = 0.6 * diversity + 0.4 * density
    
    Why this formula?
        A question touching 4 domains with just 1 keyword each is harder
        (breadth) than one touching 1 domain with 10 keywords (depth).
        The 60/40 split encodes this priority.
    """
    diversity = min(n_domains / 4.0, 1.0)
    density   = 1.0 - math.exp(-total_hits / 6.0)
    return 0.6 * diversity + 0.4 * density


# ── Apply ─────────────────────────────────────────────────────────────────────
domain_results = df['query'].apply(measure_cross_domain)
df['n_domains']          = domain_results.apply(lambda x: x[0])
df['domain_hits']        = domain_results.apply(lambda x: x[1])
df['domains_found']      = domain_results.apply(lambda x: x[2])
df['feat_cross_domain']  = df.apply(
    lambda row: normalise_cross_domain_score(row['n_domains'], row['domain_hits']),
    axis=1
)

# ── Show ──────────────────────────────────────────────────────────────────────
print('=== Feature: Cross-Domain Dependencies ===')
print(f'Scale: 0.0 = single domain | 1.0 = many domains + rich vocabulary')
print()
for _, row in df.iterrows():
    doms = ', '.join(row['domains_found']) if row['domains_found'] else '—'
    print(f"  [{row['task_type']:12}] score={row['feat_cross_domain']:.3f}  "
          f"n_domains={row['n_domains']}  domains=[{doms}]")

---
## STEP 5 — Feature: Ambiguity Score

**What it measures:** How underspecified or vague the query is.

**Why it matters:** Vague questions are harder to answer well because the solver
must resolve the ambiguity before answering — adding an implicit reasoning step.

**Components:**
- **Vague language** (`somehow`, `approximately`, `in some way`, `might`, `could`)
- **Undefined references** (`this`, `that`, `it`, `those` without clear antecedents)
- **Open-endedness** (`discuss`, `explain broadly`, `describe generally`)
- **Low information density** (inverse of compression ratio)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Feature: Ambiguity Score
# ─────────────────────────────────────────────────────────────────────────────

# Pattern groups
VAGUE_HEDGES = re.compile(
    r'\b(somehow|approximately|roughly|around|about|maybe|perhaps|possibly|'
    r'probably|might|could|should|would|tends to|in some way|generally|'
    r'typically|usually|often|sometimes|rarely|few|some|many|various|'
    r'several|certain|other|something|anything|everything)\b',
    re.IGNORECASE
)

# Pronouns without clear antecedents signal undefined references
UNDEF_REFS = re.compile(
    r'\b(this|that|it|those|these|they|them|he|she|we)\b',
    re.IGNORECASE
)

# Open-ended verbs → question has no single right answer
OPEN_ENDED = re.compile(
    r'\b(discuss|describe|explain|elaborate|explore|analyse|analyze|'
    r'comment on|reflect on|argue|debate|compare and contrast|'
    r'write about|think about)\b',
    re.IGNORECASE
)

# Negation patterns add implicit branches / ambiguity
NEGATIONS = re.compile(
    r'\b(not|no|never|neither|nor|without|except|unless|despite)\b',
    re.IGNORECASE
)


def measure_ambiguity(text: str) -> Dict[str, float]:
    """
    Compute ambiguity sub-scores and return as a dict.
    
    Returns
    -------
    dict with keys:
        hedge_score    : Vague hedge word density
        ref_score      : Undefined pronoun density  
        open_score     : Open-ended question indicator
        negation_score : Negation density
        ambiguity_total: Weighted combination
    """
    tokens = text.split()
    n_tokens = max(len(tokens), 1)
    
    # Counts (normalised by token count to avoid length bias)
    n_hedges   = len(VAGUE_HEDGES.findall(text))
    n_refs     = len(UNDEF_REFS.findall(text))
    n_open     = len(OPEN_ENDED.findall(text))
    n_neg      = len(NEGATIONS.findall(text))
    
    # Density scores (count / length, sigmoid-mapped)
    hedge_score = 1.0 - math.exp(-(n_hedges / n_tokens) * 20)
    ref_score   = 1.0 - math.exp(-(n_refs   / n_tokens) * 15)
    open_score  = min(n_open / 2.0, 1.0)   # binary: present or not
    neg_score   = 1.0 - math.exp(-(n_neg   / n_tokens) * 12)
    
    # Weighted combination
    # Open-ended verbs are the strongest signal (0.40)
    total = (
        0.40 * open_score
        + 0.30 * hedge_score
        + 0.20 * ref_score
        + 0.10 * neg_score
    )
    
    return {
        'hedge_score':     round(hedge_score, 4),
        'ref_score':       round(ref_score,   4),
        'open_score':      round(open_score,  4),
        'negation_score':  round(neg_score,   4),
        'ambiguity_total': round(total,        4),
    }


# ── Apply ─────────────────────────────────────────────────────────────────────
ambig_results = df['query'].apply(measure_ambiguity)
df['feat_ambiguity']  = ambig_results.apply(lambda x: x['ambiguity_total'])
df['ambig_hedge']     = ambig_results.apply(lambda x: x['hedge_score'])
df['ambig_ref']       = ambig_results.apply(lambda x: x['ref_score'])
df['ambig_open']      = ambig_results.apply(lambda x: x['open_score'])

# ── Show ──────────────────────────────────────────────────────────────────────
print('=== Feature: Ambiguity Score ===')
print('Scale: 0.0 = precise, well-specified | 1.0 = vague, open-ended')
print()
cols = ['id', 'task_type', 'feat_ambiguity', 'ambig_hedge', 'ambig_open']
print(df[cols].round(3).to_string(index=False))

---
## STEP 6 — Additional Features (Token Count, Compression, TTR, Hops)

These are computed together since they're all single-pass over the text.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Additional Features: Token Count, Compression Ratio, TTR, Hop Count
# ─────────────────────────────────────────────────────────────────────────────

# Multi-hop connector words (different from step words — these connect
# information ACROSS entities, not within a sequence)
HOP_SIGNALS = re.compile(
    r'\b(according to|based on|as stated in|as shown in|'
    r'therefore|consequently|it follows that|as a result|'
    r'which means|this implies|using this|from this|'
    r'who was|which was|that was|whose|of which)\b',
    re.IGNORECASE
)

QUESTION_WORDS = re.compile(
    r'\b(who|what|when|where|why|how|which|whose|whom)\b',
    re.IGNORECASE
)


def compute_basic_features(text: str) -> Dict[str, float]:
    """
    Compute all basic text features in a single pass.
    
    Features
    --------
    token_count_score   : Length normalised to [0,1] via sigmoid
    compression_score   : Kolmogorov proxy via gzip
    type_token_ratio    : Lexical diversity (unique/total words)
    sentence_score      : Number of sentences normalised
    hop_score           : Multi-hop signal density
    question_density    : Embedded sub-questions density
    """
    # ── Tokenise ──────────────────────────────────────────────────────────────
    # Simple whitespace split. For production: use tiktoken or a proper tokeniser.
    tokens = text.split()
    n_tokens = len(tokens)
    
    if n_tokens == 0:
        return {k: 0.0 for k in [
            'token_count_score', 'compression_score',
            'type_token_ratio', 'sentence_score',
            'hop_score', 'question_density',
        ]}
    
    # ── Token Count ────────────────────────────────────────────────────────────
    # Sigmoid: at 512 tokens score ≈ 0.63, at 1024 ≈ 0.86, at 2048 ≈ 0.98
    token_count_score = 1.0 - math.exp(-n_tokens / 512.0)
    
    # ── Kolmogorov Complexity Proxy ────────────────────────────────────────────
    # Theory: the complexity of a string = length of shortest program producing it.
    # Approximation: gzip compression ratio.
    #
    # Simple text ('aaaaaa') compresses to near 0 → low ratio → low complexity.
    # Information-rich text (math formulas) compresses poorly → high ratio → hard.
    raw_bytes  = text.encode('utf-8')
    comp_bytes = gzip.compress(raw_bytes, compresslevel=9)
    raw_len    = max(len(raw_bytes), 1)
    comp_ratio = len(comp_bytes) / raw_len
    # Typical range: 0.25 (very repetitive) to 0.90 (dense math)
    # Rescale to [0,1] by subtracting baseline 0.25 and dividing by 0.65
    compression_score = min(max((comp_ratio - 0.25) / 0.65, 0.0), 1.0)
    
    # ── Type-Token Ratio (TTR) ─────────────────────────────────────────────────
    # TTR = unique words / total words.
    # High TTR = diverse vocabulary = richer, harder content.
    # Note: TTR naturally falls with length (longer texts reuse words more).
    # For length-corrected TTR, use Root TTR = unique / sqrt(total).
    unique_tokens = len(set(t.lower().strip('.,;:!?"()[]{}') for t in tokens))
    ttr = unique_tokens / n_tokens                      # standard TTR
    root_ttr = unique_tokens / math.sqrt(n_tokens)      # length-corrected
    type_token_ratio = min(root_ttr / 10.0, 1.0)       # normalise root_ttr to [0,1]
    
    # ── Sentence Count ────────────────────────────────────────────────────────
    sentences = re.split(r'[.!?]+\s+', text)
    n_sentences = max(len(sentences), 1)
    sentence_score = min(n_sentences / 15.0, 1.0)      # cap at 15 sentences
    
    # ── Multi-Hop Signals ─────────────────────────────────────────────────────
    n_hops = len(HOP_SIGNALS.findall(text))
    hop_score = 1.0 - math.exp(-n_hops / 4.0)          # saturates at 4 hops
    
    # ── Embedded Question Density ─────────────────────────────────────────────
    n_questions = len(QUESTION_WORDS.findall(text))
    question_density = 1.0 - math.exp(-n_questions / 5.0)
    
    return {
        'token_count_score':  round(token_count_score,  4),
        'compression_score':  round(compression_score,  4),
        'type_token_ratio':   round(type_token_ratio,   4),
        'sentence_score':     round(sentence_score,     4),
        'hop_score':          round(hop_score,          4),
        'question_density':   round(question_density,   4),
        # Raw values for inspection
        '_n_tokens':    n_tokens,
        '_n_sentences': n_sentences,
        '_raw_ttr':     round(ttr, 4),
        '_root_ttr':    round(root_ttr, 4),
        '_n_hops':      n_hops,
    }


# ── Apply ─────────────────────────────────────────────────────────────────────
basic_results = df['query'].apply(compute_basic_features)

# Unpack all keys into separate dataframe columns
for key in [
    'token_count_score', 'compression_score', 'type_token_ratio',
    'sentence_score', 'hop_score', 'question_density',
    '_n_tokens', '_n_sentences', '_raw_ttr', '_root_ttr', '_n_hops',
]:
    feat_col = key if key.startswith('_') else f'feat_{key}'
    df[feat_col] = basic_results.apply(lambda x: x[key])

# ── Show ──────────────────────────────────────────────────────────────────────
print('=== Additional Features ===')
print()
feat_cols = ['id', 'task_type',
             'feat_token_count_score', 'feat_compression_score',
             'feat_type_token_ratio',  'feat_hop_score']
print(df[feat_cols].round(3).to_string(index=False))

print(f'\nRaw stats:')
raw_cols = ['id', '_n_tokens', '_n_sentences', '_raw_ttr', '_root_ttr', '_n_hops']
print(df[raw_cols].to_string(index=False))

---
## STEP 7 — Combine Features → Single Complexity Score

All individual features are combined via a **weighted linear combination**.

The weights below are tuned based on GAIA level labels and empirical results.
You can adjust them for your specific task distribution.

| Feature | Default Weight | Rationale |
|---|---|---|
| `step_score` | 0.18 | Most direct measure of required reasoning depth |
| `branch_score` | 0.16 | Conditional cases multiply solution complexity |
| `cross_domain` | 0.18 | Multi-domain = multi-skill requirement |
| `ambiguity` | 0.12 | Underspecification adds implicit reasoning |
| `hop_score` | 0.14 | Multi-hop = longest reasoning chains |
| `compression` | 0.10 | Information density per character |
| `ttr` | 0.06 | Vocabulary richness |
| `token_count` | 0.06 | Raw length |

**Total = 1.00**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Weighted Combination → Final Complexity Score
# ─────────────────────────────────────────────────────────────────────────────

# ──────────────────────────────────────────────────────────────────────────────
#  TUNE THESE WEIGHTS for your dataset / research question.
#  Rule: weights must sum to 1.0
# ──────────────────────────────────────────────────────────────────────────────
FEATURE_WEIGHTS: Dict[str, float] = {
    'feat_step_score':          0.18,   # reasoning step count
    'feat_branch_score':        0.16,   # decision branch depth
    'feat_cross_domain':        0.18,   # cross-domain dependency
    'feat_ambiguity':           0.12,   # ambiguity / underspecification
    'feat_hop_score':           0.14,   # multi-hop signal
    'feat_compression_score':   0.10,   # Kolmogorov proxy
    'feat_type_token_ratio':    0.06,   # lexical diversity
    'feat_token_count_score':   0.06,   # raw length
}

# Sanity check: weights must sum to 1.0
weight_sum = sum(FEATURE_WEIGHTS.values())
assert abs(weight_sum - 1.0) < 1e-6, f'Weights sum to {weight_sum:.4f}, expected 1.0'
print(f'✅ Weights sum = {weight_sum:.4f}')


def compute_complexity_score(row: pd.Series, weights: Dict[str, float]) -> float:
    """
    Weighted linear combination of normalised features.
    
    All input features are already in [0, 1].
    Output is guaranteed to be in [0, 1].
    """
    score = sum(
        weights[feat] * row[feat]
        for feat in weights
        if feat in row and pd.notna(row[feat])
    )
    return round(min(max(score, 0.0), 1.0), 4)


def assign_difficulty_tier(score: float) -> str:
    """
    Map complexity score to a 3-tier label.
    
    Thresholds are calibrated to GAIA levels:
        EASY   : < 0.35  (Level 1 in GAIA)
        MEDIUM : 0.35–0.70  (Level 2)
        HARD   : >= 0.70  (Level 3)
    """
    if score < 0.35:
        return 'EASY'
    elif score < 0.70:
        return 'MEDIUM'
    else:
        return 'HARD'


# ── Compute final scores ──────────────────────────────────────────────────────
df['complexity_score'] = df.apply(
    lambda row: compute_complexity_score(row, FEATURE_WEIGHTS),
    axis=1
)
df['difficulty_tier'] = df['complexity_score'].apply(assign_difficulty_tier)

# Per-feature weighted contribution (useful for explainability)
for feat, weight in FEATURE_WEIGHTS.items():
    contrib_col = f'contrib_{feat.replace("feat_", "")}'
    df[contrib_col] = (df[feat] * weight).round(4)

# ── Show ──────────────────────────────────────────────────────────────────────
print()
print('=== Final Complexity Scores ===')
print(f'Scale: 0.0 (trivial) → 1.0 (extremely hard)')
print(f'Tiers: EASY < 0.35 | MEDIUM 0.35–0.70 | HARD >= 0.70')
print()

result_cols = ['id', 'task_type', 'complexity_score', 'difficulty_tier']
display_result = df[result_cols].sort_values('complexity_score', ascending=False)
print(display_result.to_string(index=False))

print(f'\n=== Score Statistics ===')
print(f'  Mean : {df["complexity_score"].mean():.4f}')
print(f'  Std  : {df["complexity_score"].std():.4f}')
print(f'  Min  : {df["complexity_score"].min():.4f}')
print(f'  Max  : {df["complexity_score"].max():.4f}')
print(f'\nDifficulty Distribution:')
print(df['difficulty_tier'].value_counts().sort_index().to_string())

---
## STEP 8 — Feature Correlation & Importance Analysis

Before trusting the scores, inspect:
1. **Correlation matrix** — are any two features so correlated that one is redundant?
2. **Feature contribution breakdown** per sample — explainability
3. **Score distribution per task type** — do the scores make intuitive sense?

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Feature Analysis & Explainability
# ─────────────────────────────────────────────────────────────────────────────

feature_cols = list(FEATURE_WEIGHTS.keys())

# ── 1. Pearson correlation matrix ─────────────────────────────────────────────
print('=== Feature Correlation Matrix ===')
print('(Values close to ±1.0 = redundant; close to 0.0 = independent)')
print()
corr_matrix = df[feature_cols].corr().round(2)

# Pretty-print (no matplotlib needed)
short_names = [c.replace('feat_', '').replace('_score', '')[:12] for c in feature_cols]
header = ''.join(f'{n:>13}' for n in short_names)
print(f'{"":>18}{header}')
for i, row_name in enumerate(short_names):
    row_vals = ''.join(f'{corr_matrix.iloc[i, j]:>13.2f}' for j in range(len(short_names)))
    print(f'{row_name:>18}{row_vals}')

# ── 2. Mean score per task type ────────────────────────────────────────────────
print(f'\n=== Mean Complexity Score by Task Type ===')
task_stats = df.groupby('task_type').agg(
    n_samples=('complexity_score', 'count'),
    mean_score=('complexity_score', 'mean'),
    std_score=('complexity_score', 'std'),
    min_score=('complexity_score', 'min'),
    max_score=('complexity_score', 'max'),
).round(4).sort_values('mean_score', ascending=False)
print(task_stats.to_string())

# ── 3. Per-feature contribution breakdown for top 3 hardest samples ───────────
print(f'\n=== Feature Contribution Breakdown (Top 3 Hardest Samples) ===')
print('(Shows which features drive the complexity score for each sample)')
print()

hardest = df.nlargest(3, 'complexity_score')
contrib_cols = [c for c in df.columns if c.startswith('contrib_')]

for _, row in hardest.iterrows():
    print(f"  Sample: {row['id']}  [{row['task_type']}]")
    print(f"  Query: {row['query'][:80]}...")
    print(f"  Final score: {row['complexity_score']:.4f}  →  {row['difficulty_tier']}")
    print(f"  Feature contributions (sorted):")
    contribs = {
        col.replace('contrib_', ''): row[col]
        for col in contrib_cols
    }
    for feat, val in sorted(contribs.items(), key=lambda x: -x[1]):
        bar = '█' * int(val * 80)
        print(f"    {feat:25} {val:.4f}  {bar}")
    print()

---
## STEP 9 — Validate Against Ground-Truth Labels

If your dataset has ground-truth difficulty labels (like GAIA levels 1/2/3
or MMLU difficulty fields), compute **rank correlation** between
your predicted complexity scores and the ground truth.

**Spearman's ρ** is used (not Pearson) because difficulty is ordinal.
- ρ > 0.70 → good agreement
- ρ > 0.50 → moderate agreement
- ρ < 0.30 → poor — consider re-tuning weights

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Validation Against Ground-Truth Difficulty Labels
# ─────────────────────────────────────────────────────────────────────────────

def spearman_rho(x: List[float], y: List[float]) -> float:
    """
    Compute Spearman rank correlation WITHOUT scipy.
    
    Spearman's ρ = Pearson correlation of the *ranks* of x and y.
    
    Why Spearman and not Pearson?
        Difficulty labels are ordinal (easy < medium < hard), not interval.
        Spearman makes no assumptions about the spacing between levels.
    """
    assert len(x) == len(y), 'x and y must have the same length'
    n = len(x)
    if n < 3:
        return float('nan')
    
    def rank(lst):
        sorted_lst = sorted(enumerate(lst), key=lambda t: t[1])
        ranks = [0.0] * len(lst)
        for r, (i, _) in enumerate(sorted_lst):
            ranks[i] = r + 1
        return ranks
    
    rx, ry = rank(x), rank(y)
    mean_rx = sum(rx) / n
    mean_ry = sum(ry) / n
    
    num   = sum((rx[i] - mean_rx) * (ry[i] - mean_ry) for i in range(n))
    den_x = math.sqrt(sum((rx[i] - mean_rx)**2 for i in range(n)))
    den_y = math.sqrt(sum((ry[i] - mean_ry)**2 for i in range(n)))
    
    if den_x == 0 or den_y == 0:
        return float('nan')
    
    return round(num / (den_x * den_y), 4)


# ── Simulate ground-truth labels for demonstration ────────────────────────────
# In production: replace this with the actual 'difficulty' column from your dataset.
# We create a proxy: map known task characteristics to a ground-truth score.
GT_MAP = {
    'm001': 0.05, 'm002': 0.10, 'm003': 0.25,   # math: easy → hard
    'm004': 0.95, 'm005': 0.80, 'm006': 0.90,
    'q001': 0.05, 'q002': 0.05,                   # simple QA
    'q003': 0.50, 'q004': 0.55,                   # medium multi-hop
    'q005': 0.90, 'q006': 0.95,                   # hard multi-hop
    'c001': 0.10,                                  # trivial code
    'c002': 0.60, 'c003': 0.95,                   # real issues
    's001': 0.05, 's002': 0.85,                   # science QA
    'g001': 0.05, 'g002': 0.90,                   # GAIA
}
df['ground_truth_score'] = df['id'].map(GT_MAP)

# ── Compute Spearman ρ ────────────────────────────────────────────────────────
valid = df.dropna(subset=['ground_truth_score'])
rho = spearman_rho(
    valid['complexity_score'].tolist(),
    valid['ground_truth_score'].tolist()
)

print('=== Validation Against Ground-Truth Difficulty ===')
print(f'\nSpearman rank correlation ρ = {rho:.4f}')
if rho > 0.70:
    print('✅  Good agreement (ρ > 0.70) — feature weights are well-calibrated')
elif rho > 0.50:
    print('⚠️   Moderate agreement (ρ > 0.50) — consider re-tuning feature weights')
else:
    print('❌  Poor agreement (ρ < 0.50) — features or weights need revision')

# ── Per-sample comparison ─────────────────────────────────────────────────────
print(f'\nPer-sample comparison (predicted vs ground truth):')
comparison = valid[['id', 'task_type', 'complexity_score', 'ground_truth_score', 'difficulty_tier']].copy()
comparison['error'] = (comparison['complexity_score'] - comparison['ground_truth_score']).abs().round(4)
comparison = comparison.sort_values('ground_truth_score', ascending=False)
print(comparison.to_string(index=False))

print(f'\nMean Absolute Error (MAE): {comparison["error"].mean():.4f}')

---
## STEP 10 — ASCII Visualisations

Visualise the distribution and per-task breakdown without any plotting library.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — ASCII Visualisations (no matplotlib needed)
# ─────────────────────────────────────────────────────────────────────────────

def ascii_bar(value: float, width: int = 40, fill: str = '█', empty: str = '░') -> str:
    """Render a horizontal bar chart as ASCII."""
    filled = int(value * width)
    return fill * filled + empty * (width - filled)


def ascii_histogram(values: List[float], n_bins: int = 10, width: int = 35) -> str:
    """Render a histogram of values in [0, 1] as ASCII."""
    bins = [0] * n_bins
    for v in values:
        idx = min(int(v * n_bins), n_bins - 1)
        bins[idx] += 1
    max_count = max(bins) if bins else 1
    lines = []
    for i, count in enumerate(bins):
        lo = i / n_bins
        hi = (i + 1) / n_bins
        bar = '█' * int((count / max_count) * width)
        tier = 'EASY  ' if hi <= 0.35 else ('MEDIUM' if hi <= 0.70 else 'HARD  ')
        lines.append(f'  {lo:.1f}–{hi:.1f} [{tier}] {bar} ({count})')
    return '\n'.join(lines)


# ── 1. Complexity score distribution ──────────────────────────────────────────
print('━' * 62)
print(' COMPLEXITY SCORE DISTRIBUTION')
print('━' * 62)
print(ascii_histogram(df['complexity_score'].tolist()))

# ── 2. All samples ranked by score ────────────────────────────────────────────
print()
print('━' * 62)
print(' ALL SAMPLES — RANKED BY COMPLEXITY')
print('━' * 62)
sorted_df = df.sort_values('complexity_score', ascending=False)
for _, row in sorted_df.iterrows():
    bar   = ascii_bar(row['complexity_score'], width=30)
    tier  = row['difficulty_tier']
    score = row['complexity_score']
    tid   = f"{row['id']:6}"
    ttype = f"{row['task_type']:12}"
    print(f'  {tid} {ttype} {score:.3f} {bar} {tier}')

# ── 3. Feature importance heatmap (mean contribution per task type) ────────────
print()
print('━' * 62)
print(' FEATURE CONTRIBUTION HEATMAP BY TASK TYPE')
print(' (darker = higher mean contribution)')
print('━' * 62)

short_feats = [
    ('step',        'contrib_step_score'),
    ('branch',      'contrib_branch_score'),
    ('cross_dom',   'contrib_cross_domain'),
    ('ambiguity',   'contrib_ambiguity'),
    ('hop',         'contrib_hop_score'),
    ('compress',    'contrib_compression_score'),
    ('ttr',         'contrib_type_token_ratio'),
    ('tok_len',     'contrib_token_count_score'),
]

task_types = df['task_type'].unique().tolist()
header = ' ' * 16 + ''.join(f'{n:>10}' for n, _ in short_feats)
print(header)
for ttype in sorted(task_types):
    sub = df[df['task_type'] == ttype]
    row_str = f'{ttype:16}'
    for _, col in short_feats:
        if col in sub.columns:
            mean_val = sub[col].mean()
            # Map [0, 0.2] → heat symbols
            if mean_val > 0.15:    sym = '  ██████'
            elif mean_val > 0.10:  sym = '  ████  '
            elif mean_val > 0.06:  sym = '  ██    '
            elif mean_val > 0.03:  sym = '  █     '
            else:                  sym = '  ░     '
            row_str += f'{sym}'
        else:
            row_str += f'{"  ?":>10}'
    print(row_str)

---
## STEP 11 — Export Enriched Dataset

Export the full enriched dataframe with all features + complexity scores
back to Parquet (or CSV). This enriched file is what gets fed into the
routing module next.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Export Enriched Dataset
# ─────────────────────────────────────────────────────────────────────────────

# ── Select columns to export ──────────────────────────────────────────────────
# Keep: original columns + complexity results + all features
ORIGINAL_COLS = ['id', 'query', 'task_type', 'dataset', 'split', 'reference_answer']
original_cols_present = [c for c in ORIGINAL_COLS if c in df.columns]

COMPLEXITY_COLS = ['complexity_score', 'difficulty_tier']
FEATURE_COLS    = [c for c in df.columns if c.startswith('feat_')]
CONTRIB_COLS    = [c for c in df.columns if c.startswith('contrib_')]
RAW_COLS        = [c for c in df.columns if c.startswith('_')]

# Full export (all columns)
export_cols = original_cols_present + COMPLEXITY_COLS + FEATURE_COLS + CONTRIB_COLS
export_df   = df[export_cols].copy()

# ── Save ──────────────────────────────────────────────────────────────────────
csv_path  = OUTPUT_DIR / 'complexity_scores.csv'
json_path = OUTPUT_DIR / 'complexity_scores.jsonl'

# CSV (always works)
export_df.to_csv(csv_path, index=False)
print(f'📄 Saved CSV  → {csv_path}')

# JSONL (for downstream systems)
with json_path.open('w', encoding='utf-8') as fh:
    for record in export_df.to_dict(orient='records'):
        fh.write(json.dumps(record) + '\n')
print(f'📝 Saved JSONL → {json_path}')

# Parquet (if pyarrow available)
try:
    parquet_path = OUTPUT_DIR / 'complexity_scores.parquet'
    export_df.to_parquet(parquet_path, index=False)
    print(f'📦 Saved Parquet → {parquet_path}')
except Exception:
    print('⚠️  Parquet skipped (install pyarrow: pip install pyarrow)')

# ── Summary report ────────────────────────────────────────────────────────────
print(f'\n{"━" * 55}')
print(f' EXPORT SUMMARY')
print(f'{"━" * 55}')
print(f'  Rows exported  : {len(export_df)}')
print(f'  Columns        : {len(export_cols)}')
print(f'  Features stored: {len(FEATURE_COLS)}')
print(f'  Contributions  : {len(CONTRIB_COLS)}')
print()

# Tier distribution in exported data
tier_counts = export_df['difficulty_tier'].value_counts()
print('  Difficulty distribution:')
for tier, count in tier_counts.items():
    pct = count / len(export_df) * 100
    bar = '█' * int(pct / 3)
    print(f'    {tier:8}: {count:3d} ({pct:4.1f}%)  {bar}')

print(f'\n  Complexity score range: '
      f'{export_df["complexity_score"].min():.4f} '
      f'– {export_df["complexity_score"].max():.4f}')
print(f'  Mean score: {export_df["complexity_score"].mean():.4f}')

print(f'\n✅ Done. Enriched dataset ready for routing pipeline.')

---
## STEP 12 — Reusable API: `ComplexityMeasurer` class

Everything above is now packaged into a clean, reusable class.
This is what you import in your production pipeline.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Production-Ready ComplexityMeasurer Class
# ─────────────────────────────────────────────────────────────────────────────

class ComplexityMeasurer:
    """
    Production-ready structural/linguistic complexity estimator.
    
    Encapsulates all features from this notebook into a single
    callable class that processes a Parquet/CSV dataset and
    returns an enriched DataFrame with complexity scores.
    
    Usage
    -----
    measurer = ComplexityMeasurer()
    enriched_df = measurer.fit_transform('data/raw/my_dataset.parquet')
    
    # Or on an existing DataFrame:
    score = measurer.score_single("Prove Fermat's Last Theorem.")
    """
    
    DEFAULT_WEIGHTS = {
        'step':         0.18,
        'branch':       0.16,
        'cross_domain': 0.18,
        'ambiguity':    0.12,
        'hop':          0.14,
        'compression':  0.10,
        'ttr':          0.06,
        'token_count':  0.06,
    }
    
    TIER_THRESHOLDS = {'EASY': 0.35, 'MEDIUM': 0.70}
    
    def __init__(self, weights: Optional[Dict[str, float]] = None):
        self.weights = weights or self.DEFAULT_WEIGHTS
        assert abs(sum(self.weights.values()) - 1.0) < 1e-6, 'Weights must sum to 1.0'
    
    def score_single(self, text: str) -> Dict[str, float]:
        """Score a single text string. Returns dict with all features + final score."""
        features = {}
        
        # All feature computations (same functions defined above)
        step_raw, _   = count_reasoning_steps(text)
        branch_w, _   = measure_decision_depth(text)
        n_dom, n_hits, doms = measure_cross_domain(text)
        ambig         = measure_ambiguity(text)
        basic         = compute_basic_features(text)
        
        features['step']         = normalise_step_score(step_raw)
        features['branch']       = normalise_branch_score(branch_w)
        features['cross_domain'] = normalise_cross_domain_score(n_dom, n_hits)
        features['ambiguity']    = ambig['ambiguity_total']
        features['hop']          = basic['hop_score']
        features['compression']  = basic['compression_score']
        features['ttr']          = basic['type_token_ratio']
        features['token_count']  = basic['token_count_score']
        
        # Weighted combination
        score = sum(self.weights[k] * v for k, v in features.items())
        score = round(min(max(score, 0.0), 1.0), 4)
        
        # Assign tier
        if score < self.TIER_THRESHOLDS['EASY']:
            tier = 'EASY'
        elif score < self.TIER_THRESHOLDS['MEDIUM']:
            tier = 'MEDIUM'
        else:
            tier = 'HARD'
        
        return {
            'complexity_score': score,
            'difficulty_tier':  tier,
            'domains_detected': doms,
            **{f'feat_{k}': round(v, 4) for k, v in features.items()},
        }
    
    def fit_transform(self, path: str) -> pd.DataFrame:
        """Load dataset from path, score all rows, return enriched DataFrame."""
        df_local = load_dataset(Path(path))
        results  = df_local['query'].apply(self.score_single)
        result_df = pd.DataFrame(results.tolist())
        return pd.concat([df_local, result_df], axis=1)
    
    def batch_score(self, texts: List[str]) -> pd.DataFrame:
        """Score a list of texts without a file. Returns DataFrame."""
        return pd.DataFrame([self.score_single(t) for t in texts])


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEMO: use the class on fresh queries
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

measurer = ComplexityMeasurer()

test_queries = [
    "What is 2 + 2?",
    "Find the derivative of f(x) = sin(x^2)",
    "If the CEO of the company that acquired DeepMind in 2014 "
    "was born in London, which university in London produced the "
    "scientist who discovered the DNA double helix using X-ray "
    "crystallography, and what department does that university's AI lab fall under?",
    "Prove that every even perfect number greater than 6 is of the form "
    "2^(p-1)(2^p - 1) where 2^p - 1 is a Mersenne prime. Additionally, "
    "determine whether odd perfect numbers can exist and relate this to "
    "the Riemann hypothesis.",
]

print('=== ComplexityMeasurer — Live Demo ===')
print()
results_df = measurer.batch_score(test_queries)

for i, (query, (_, row)) in enumerate(zip(test_queries, results_df.iterrows())):
    print(f'Query {i+1}: "{query[:70]}..."' if len(query) > 70 else f'Query {i+1}: "{query}"')
    print(f'  Score    : {row["complexity_score"]:.4f}')
    print(f'  Tier     : {row["difficulty_tier"]}')
    print(f'  Domains  : {row["domains_detected"]}')
    feat_items = [(k, v) for k, v in row.items() if k.startswith('feat_')]
    feat_items.sort(key=lambda x: -x[1])
    print(f'  Top feats: ' + ', '.join(f'{k.replace("feat_","")}={v:.3f}' for k, v in feat_items[:4]))
    print()